# non-diff-fn-wrap — worked example 2: Chain of non-differentiable ops stays isolated

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `non-diff-fn-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a non-differentiable op produces a `MiniTensor` with `recipe=None`, any further non-differentiable op applied to that output also produces `recipe=None`. The isolation is automatic: because neither input has `requires_grad=True`, the three-gate AND evaluates to False at every step, and no Recipe is ever created. The entire non-differentiable branch remains invisible to the reverse pass.

## Worked solution

**Step 1 — Build two non-differentiable wrappers.**
We wrap `torch.floor` and `torch.sign` with `is_differentiable=False`. Neither will attach a Recipe.

**Step 2 — Build a tracked leaf tensor.**
We create a `MiniTensor` with `requires_grad=True`. This is the input that would normally propagate gradients if the ops were differentiable.

**Step 3 — Apply floor, then sign.**
`mask = floor_wrap(x)` — because `is_differentiable=False`, `mask.requires_grad=False` and `mask.recipe=None`. Then `idx = sign_wrap(mask)` — again non-differentiable, and now the input also has `requires_grad=False`, so the gate fails doubly.

**Step 4 — Verify isolation at each step.**
At each node we confirm `requires_grad=False` and `recipe=None`. We also confirm the original tracked tensor `x` is untouched.

In [ ]:
import torch

# ---- Minimal scaffold ----
class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args
        self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = None

grad_tracking_enabled = True

def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

# Non-differentiable wrappers
floor_wrap = wrap_forward_fn(torch.floor, is_differentiable=False)
sign_wrap  = wrap_forward_fn(torch.sign,  is_differentiable=False)
# Differentiable control
mul_wrap   = wrap_forward_fn(torch.mul)

torch.manual_seed(17)
x = MiniTensor(torch.randn(5), requires_grad=True)

# Non-diff chain
mask = floor_wrap(x)
idx  = sign_wrap(mask)

# Differentiable side branch
c = mul_wrap(x, x)

print('--- non-diff chain ---')
print(f'mask requires_grad: {mask.requires_grad}')  # False
print(f'mask recipe      : {mask.recipe}')          # None
print(f'idx  requires_grad: {idx.requires_grad}')   # False
print(f'idx  recipe      : {idx.recipe}')           # None
print('--- diff side ---')
print(f'c requires_grad  : {c.requires_grad}')      # True
print(f'c recipe set     : {c.recipe is not None}') # True

assert mask.requires_grad is False and mask.recipe is None
assert idx.requires_grad  is False and idx.recipe  is None
assert c.requires_grad    is True  and c.recipe    is not None